In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
def split_folder_80_20_abs(
    in_path: str,
    project_root: str = "/home/lisa/Arupreza/UIDS/UIDS-II/Aljabri Light Adaptive IDS",
    train_ratio: float = 0.8,
    sort_by: str | None = "Time_Offset",
    pattern: str = "*.csv",
    verbose: bool = True,
):
    """
    Read CSVs from `in_path` (relative or absolute),
    write per-file 80/20 splits into:
        {project_root}/TrainSplit/{vehicle}/
        {project_root}/TestSplit/{vehicle}/
    """

    root = Path(project_root)

    # ---- Resolve input path robustly ----
    p = Path(in_path)
    candidates = [p]
    if not p.is_absolute():
        candidates.append(root / p)

    in_dir = None
    for c in candidates:
        if c.exists():
            in_dir = c
            break

    if in_dir is None:
        raise FileNotFoundError(
            "Input path does not exist. Tried:\n"
            + "\n".join([f" - {c}" for c in candidates])
        )

    vehicle = in_dir.name  # e.g., "Kia"

    out_train_dir = root / "TrainSplit" / vehicle
    out_test_dir  = root / "TestSplit" / vehicle
    out_train_dir.mkdir(parents=True, exist_ok=True)
    out_test_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(in_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No CSV files matched '{pattern}' in {in_dir}")

    summary = []
    for fp in files:
        df = pd.read_csv(fp)
        if df.empty:
            if verbose:
                print(f"[SKIP] {fp.name}: empty")
            continue

        if sort_by is not None and sort_by in df.columns:
            df = df.sort_values(sort_by, kind="mergesort").reset_index(drop=True)
        else:
            df = df.reset_index(drop=True)

        n = len(df)
        cut = int(n * train_ratio)

        # keep both parts if possible
        if n >= 2:
            cut = max(1, min(cut, n - 1))
        else:
            cut = 1

        df_train = df.iloc[:cut].copy()
        df_test  = df.iloc[cut:].copy()

        (out_train_dir / fp.name).write_text(df_train.to_csv(index=False))
        (out_test_dir / fp.name).write_text(df_test.to_csv(index=False))

        summary.append((fp.name, n, len(df_train), len(df_test)))
        if verbose:
            print(f"[OK] {fp.name}: total={n}, train={len(df_train)}, test={len(df_test)}")

    if verbose and summary:
        tot_train = sum(r[2] for r in summary)
        tot_test  = sum(r[3] for r in summary)
        print(f"\nInput used: {in_dir}")
        print(f"Saved train splits to: {out_train_dir}")
        print(f"Saved test splits  to: {out_test_dir}")
        print(f"TOTAL rows -> train={tot_train}, test={tot_test}")

    return str(out_train_dir), str(out_test_dir), summary

In [11]:
PATH = "/home/lisa/Arupreza/UIDS/UIDS-II/Split_data/Train/Kia"  # relative is fine now
train_dir, test_dir, split_summary = split_folder_80_20_abs(PATH)

[OK] Kia_AF.csv: total=600000, train=480000, test=120000
[OK] Kia_DoS_H.csv: total=532000, train=425600, test=106400
[OK] Kia_DoS_L.csv: total=448000, train=358400, test=89600
[OK] Kia_DoS_LL.csv: total=432670, train=346136, test=86534
[OK] Kia_DoS_M.csv: total=489999, train=391999, test=98000
[OK] Kia_DoS_R.csv: total=483253, train=386602, test=96651
[OK] Kia_Fuzz_H.csv: total=532000, train=425600, test=106400
[OK] Kia_Fuzz_L.csv: total=448000, train=358400, test=89600
[OK] Kia_Fuzz_LL.csv: total=432726, train=346180, test=86546
[OK] Kia_Fuzz_M.csv: total=489999, train=391999, test=98000
[OK] Kia_Fuzz_R.csv: total=483037, train=386429, test=96608
[OK] Kia_Rep_H.csv: total=532000, train=425600, test=106400
[OK] Kia_Rep_L.csv: total=448000, train=358400, test=89600
[OK] Kia_Rep_LL.csv: total=432670, train=346136, test=86534
[OK] Kia_Rep_M.csv: total=489999, train=391999, test=98000
[OK] Kia_Rep_R.csv: total=482764, train=386211, test=96553

Input used: /home/lisa/Arupreza/UIDS/UIDS-II/S

In [17]:
from pathlib import Path
import os, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, classification_report

from imblearn.over_sampling import RandomOverSampler, SMOTE

# --- Your dataset schema ---
FEATURE_COLS = [
    "Time_Offset", "CAN_ID", "Data_Length",
    "One","Two","Three","Four","Five","Six","Seven","Eight"
]
LABEL_COL = "Label"

PROJECT_ROOT = Path("/home/lisa/Arupreza/UIDS/UIDS-II/Aljabri Light Adaptive IDS")

# You said you already created these:
TRAIN_SPLIT_DIR = PROJECT_ROOT / "TrainSplit" / "Kia"
TEST_SPLIT_DIR  = PROJECT_ROOT / "TestSplit"  / "Kia"

# To match the journal 70/15/15 split, we combine ALL your available split files,
# then we re-split 70/15/15 from the union (random, stratified).
DATA_DIRS_FOR_JOURNAL_SPLIT = [TRAIN_SPLIT_DIR, TEST_SPLIT_DIR]

In [18]:
def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def parse_can_id(x):
    # handles int/float/"0x.."/hex string
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, (float, np.floating)):
        return int(x)
    s = str(x).strip()
    if s == "":
        return np.nan
    try:
        return int(s)
    except Exception:
        pass
    s2 = s.lower()
    if s2.startswith("0x"):
        try:
            return int(s2, 16)
        except Exception:
            return np.nan
    if any(c in "abcdef" for c in s2):
        try:
            return int(s2, 16)
        except Exception:
            return np.nan
    return np.nan

def load_concat_csvs(folders, pattern="*.csv"):
    dfs = []
    for folder in folders:
        folder = Path(folder)
        if not folder.exists():
            raise FileNotFoundError(f"Folder not found: {folder}")
        files = sorted(folder.glob(pattern))
        if not files:
            raise FileNotFoundError(f"No CSV files in: {folder}")
        for fp in files:
            df = pd.read_csv(fp)
            if df.empty:
                continue
            missing = [c for c in (FEATURE_COLS + [LABEL_COL]) if c not in df.columns]
            if missing:
                raise ValueError(f"{fp.name} missing columns: {missing}")
            df["_source_file"] = fp.name
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

def preprocess_df(df):
    df = df.copy()
    df["CAN_ID"] = df["CAN_ID"].apply(parse_can_id)
    for c in FEATURE_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0.0)
    df[LABEL_COL] = df[LABEL_COL].astype(str)
    return df

# paper: MinMax [0,1] then StandardScaler :contentReference[oaicite:10]{index=10}
def apply_paper_preprocessing(X_train, X_val, X_test):
    mm = MinMaxScaler(feature_range=(0, 1))
    X_train_mm = mm.fit_transform(X_train)
    X_val_mm   = mm.transform(X_val)
    X_test_mm  = mm.transform(X_test)

    ss = StandardScaler()
    X_train_fin = ss.fit_transform(X_train_mm)
    X_val_fin   = ss.transform(X_val_mm)
    X_test_fin  = ss.transform(X_test_mm)
    return X_train_fin, X_val_fin, X_test_fin, mm, ss

In [19]:
# Architecture per paper: 128, 64, softmax; ReLU in hidden layers :contentReference[oaicite:11]{index=11}
class DeepMLP(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.softmax(self.fc3(x))  # probabilities (to match paper description)
        return x

# Categorical Crossentropy: -sum(y * log(p)) :contentReference[oaicite:12]{index=12}
def categorical_crossentropy(probs, y_onehot, eps=1e-12):
    probs = torch.clamp(probs, eps, 1.0)
    loss = -(y_onehot * torch.log(probs)).sum(dim=1).mean()
    return loss

class TabularDataset(Dataset):
    def __init__(self, X, y_int, num_classes):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y_int, dtype=torch.long)
        self.num_classes = num_classes

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
set_seed(42)

# 1) Load all your available Kia CSVs (from TrainSplit + TestSplit) then preprocess
df_all = preprocess_df(load_concat_csvs(DATA_DIRS_FOR_JOURNAL_SPLIT))
print("Total rows:", len(df_all))
print("Label distribution:\n", df_all[LABEL_COL].value_counts())

X = df_all[FEATURE_COLS].values.astype(np.float32)
y_str = df_all[LABEL_COL].values.astype(str)

# 2) Journal split: 70% train, 15% val, 15% test :contentReference[oaicite:13]{index=13}
# Do it in two steps to get exact ratios:
X_train, X_tmp, y_train_str, y_tmp_str = train_test_split(
    X, y_str, test_size=0.30, random_state=42, stratify=y_str
)
X_val, X_test, y_val_str, y_test_str = train_test_split(
    X_tmp, y_tmp_str, test_size=0.50, random_state=42, stratify=y_tmp_str
)
print("Split sizes:", len(X_train), len(X_val), len(X_test))

# 3) Label encode (fit on train only; enforce no unseen labels)
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)
y_val   = le.transform(y_val_str)

unseen = set(y_test_str) - set(le.classes_)
if unseen:
    raise ValueError(f"Unseen labels in test not present in train: {sorted(unseen)}")
y_test  = le.transform(y_test_str)

num_classes = len(le.classes_)
in_dim = X_train.shape[1]
print("in_dim:", in_dim, "num_classes:", num_classes, "classes:", list(le.classes_))

# 4) Class imbalance handling (paper discusses SMOTE and Random Oversampling) :contentReference[oaicite:14]{index=14}
IMBALANCE_STRATEGY = "smote"   # "smote" or "random_over" or "none"

if IMBALANCE_STRATEGY == "smote":
    # SMOTE needs enough samples per class; if it errors, switch to random_over
    sm = SMOTE(random_state=42, k_neighbors=5)
    X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)
elif IMBALANCE_STRATEGY == "random_over":
    ros = RandomOverSampler(random_state=42)
    X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)
else:
    X_train_bal, y_train_bal = X_train, y_train

print("Train label counts (after balance):", np.bincount(y_train_bal))

# 5) Paper preprocessing: MinMax -> StandardScaler :contentReference[oaicite:15]{index=15}
X_train_pp, X_val_pp, X_test_pp, mm, ss = apply_paper_preprocessing(
    X_train_bal, X_val, X_test
)

# 6) DataLoaders (paper batch size 32) :contentReference[oaicite:16]{index=16}
batch_size = 32
train_loader = DataLoader(TabularDataset(X_train_pp, y_train_bal, num_classes), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TabularDataset(X_val_pp,   y_val,       num_classes), batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(TabularDataset(X_test_pp,  y_test,      num_classes), batch_size=batch_size, shuffle=False)

# 7) Model + Adam (paper) :contentReference[oaicite:17]{index=17}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepMLP(in_dim=in_dim, num_classes=num_classes).to(device)

# Parameter count print (paper reports 10,117 for their setting) :contentReference[oaicite:18]{index=18}
param_count = sum(p.numel() for p in model.parameters())
print("Model parameters:", param_count)

# Paper does not explicitly print LR in the cited section; keep it configurable.
lr = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# 8) Training (paper: 10 epochs) :contentReference[oaicite:19]{index=19}
epochs = 10

def eval_loader(model, loader):
    model.eval()
    all_true, all_pred = [], []
    total_loss = 0.0
    n = 0
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(device)
            yb = yb.to(device)
            probs = model(Xb)

            y_onehot = torch.zeros((yb.shape[0], num_classes), device=device)
            y_onehot.scatter_(1, yb.view(-1, 1), 1.0)

            loss = categorical_crossentropy(probs, y_onehot)
            total_loss += loss.item() * yb.shape[0]
            n += yb.shape[0]

            pred = torch.argmax(probs, dim=1)
            all_true.append(yb.cpu().numpy())
            all_pred.append(pred.cpu().numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    avg_loss = total_loss / max(1, n)
    acc = accuracy_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return avg_loss, acc, rec, f1, y_true, y_pred

best_val_f1 = -1.0
best_state = None

for ep in range(1, epochs + 1):
    model.train()
    running = 0.0
    n = 0
    for Xb, yb in train_loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        probs = model(Xb)
        y_onehot = torch.zeros((yb.shape[0], num_classes), device=device)
        y_onehot.scatter_(1, yb.view(-1, 1), 1.0)

        loss = categorical_crossentropy(probs, y_onehot)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running += loss.item() * yb.shape[0]
        n += yb.shape[0]

    train_loss = running / max(1, n)
    val_loss, val_acc, val_rec, val_f1, _, _ = eval_loader(model, val_loader)

    print(f"Epoch {ep:02d}/{epochs} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f} "
          f"| val_acc={val_acc:.4f} val_rec={val_rec:.4f} val_f1={val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

# 9) Test evaluation (accuracy/recall/F1 + confusion matrix) :contentReference[oaicite:20]{index=20}
model.load_state_dict(best_state)
test_loss, test_acc, test_rec, test_f1, y_true, y_pred = eval_loader(model, test_loader)

print("\n=== TEST (best val_f1) ===")
print(f"test_loss={test_loss:.6f}")
print(f"Accuracy={test_acc:.4f} | Recall(macro)={test_rec:.4f} | F1(macro)={test_f1:.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))

Total rows: 7757117
Label distribution:
 Label
Normal    6900368
DoS        285838
Fuzz       285755
Replay     285156
Name: count, dtype: int64
Split sizes: 5429981 1163568 1163568
in_dim: 11 num_classes: 4 classes: [np.str_('DoS'), np.str_('Fuzz'), np.str_('Normal'), np.str_('Replay')]
Train label counts (after balance): [4830257 4830257 4830257 4830257]
Model parameters: 10052


In [ ]:
import joblib

SAVE_DIR = PROJECT_ROOT / "artifacts_deepmlp_kia"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), SAVE_DIR / "deepmlp_best.pt")
joblib.dump(mm, SAVE_DIR / "minmax.joblib")
joblib.dump(ss, SAVE_DIR / "standard.joblib")
joblib.dump(le, SAVE_DIR / "label_encoder.joblib")

print("Saved to:", SAVE_DIR)